# Oturum 4 — Martini 3 Input Hazırlama

**Biyofizik 2026 Kursu · 25 Ağustos 2026 · Dr. Öğr. Üyesi Ekrem Yaşar**

Bu notebook'ta, sabah CHARMM-GUI ile all-atom olarak hazırladığımız **aynı proteini**
(`6JOD` chain A — Anjiyotensin II tip-2 reseptörü) bu kez **terminalden** ve
**Martini 3 kaba-taneli** olarak hazırlayacağız.

> ⚠️ **Uzun simülasyon koşmuyoruz.** Amacımız simülasyona *girecek* sistemi kurmak.

**Akış:**
1. Kurulum
2. Yapıyı hazırla (chain A'yı ayıkla)
3. `martinize2` → proteini kaba-tanele
4. `insane` → membran + su + iyon
5. Topolojiyi düzelt
6. `gmx grompp` → doğrula
7. *(opsiyonel)* kısa enerji minimizasyonu


---
## 1 · Kurulum

Bu hücre ~3–5 dakika sürer. Bir kez çalıştırmanız yeterli.

| Paket | Ne için |
|---|---|
| `vermouth` | `martinize2` komutunu sağlar |
| `insane` | membran/kutu inşası |
| `gromacs` | `gmx grompp` ile doğrulama |
| `dssp` | ikincil yapı tayini (`mkdssp`) |


In [ ]:
%%capture
!pip install -q vermouth insane
!apt-get -qq update
!apt-get -qq install -y gromacs dssp


In [ ]:
# Kurulum kontrolu
!martinize2 --version 2>&1 | head -2
!insane --help 2>&1 | head -3
!gmx --version 2>&1 | grep -i 'GROMACS version'
!which mkdssp || echo 'mkdssp bulunamadi -> asagida -ss alternatifini kullanacagiz'


---
## 2 · Yapıyı hazırla

`6JOD`'u indirip **sadece chain A**'yı (AT2R) ayıklıyoruz.

> 🎓 Hatırlatma: bu yapıda C zinciri **BRIL füzyonu**, H ve L zincirleri **Fab fragmanı**.
> Hiçbiri hücrede yok — simülasyona koymuyoruz.

Bu oturumda ligandı (chain B) da almıyoruz; sade bir protein+membran sistemiyle başlıyoruz.


In [ ]:
!wget -q https://files.rcsb.org/download/6JOD.pdb -O 6jod.pdb

# Sadece chain A'nin ATOM satirlari (su ve hetero gruplar haric)
kept = []
for line in open('6jod.pdb'):
    if line.startswith('ATOM  ') and line[21] == 'A':
        kept.append(line)
kept.append('END\n')
open('at2r.pdb','w').writelines(kept)

resids = sorted({int(l[22:26]) for l in kept if l.startswith('ATOM')})
print(f'chain A: {len(resids)} rezidu ({resids[0]}..{resids[-1]}), {len(kept)-1} atom')


> 🔍 **Kontrol:** 306 rezidü, 35–340 arası görmelisiniz.
> Dizide 312 rezidü var — **341–346 arası çözülmemiş** (C-terminal kuyruk).
> İç loop boşluğu yok, bu yüzden modelleme yapmamıza gerek kalmıyor.


---
## 3 · `martinize2` — proteini kaba-tanele

Bu, oturumun kalbi. All-atom proteini Martini 3 bead'lerine çeviriyoruz.

| Bayrak | Ne yapar | Neden önemli |
|---|---|---|
| `-ff martini3001` | Martini 3 kuvvet alanı | Martini 2 ile karıştırmayın |
| `-dssp` | ikincil yapıyı belirler | bead tipleri ikincil yapıya bağlı |
| `-elastic` | elastic network ekler | **bunsuz protein açılır** |
| `-ef 700` | yay kuvvet sabiti (kJ/mol/nm²) | çok yüksek = taş; çok düşük = açılır |
| `-el 0.5 -eu 0.9` | yay mesafe aralığı (nm) | hangi bead çiftleri bağlanacak |


In [ ]:
!martinize2 \
  -f at2r.pdb \
  -o topol.top \
  -x at2r_cg.pdb \
  -ff martini3001 \
  -dssp \
  -elastic -ef 700 -el 0.5 -eu 0.9 -ea 0 -ep 0 \
  -maxwarn 10


### `-dssp` hata verirse

`mkdssp` bulunamazsa ikincil yapıyı elle verebilirsiniz. Aşağıdaki hücreyi
**sadece yukarısı hata verdiyse** çalıştırın — tüm proteini heliks (`H`) sayar,
bir GPCR için kaba ama kabul edilebilir bir yaklaşımdır.

> ⚠️ Bu bir kestirme yoldur. Gerçek çalışmada `mkdssp` kurun.


In [ ]:
# SADECE yukaridaki hucre -dssp yuzunden hata verdiyse calistirin
# n = 306
# !martinize2 -f at2r.pdb -o topol.top -x at2r_cg.pdb -ff martini3001 \
#   -ss $(python3 -c "print('H'*306)") \
#   -elastic -ef 700 -el 0.5 -eu 0.9 -ea 0 -ep 0 -maxwarn 10


In [ ]:
# Ne kadar kucultuk?
aa = sum(1 for l in open('at2r.pdb') if l.startswith('ATOM'))
cg = sum(1 for l in open('at2r_cg.pdb') if l.startswith(('ATOM','HETATM')))
print(f'All-atom : {aa:>6} atom')
print(f'Martini  : {cg:>6} bead')
print(f'Oran     : {aa/cg:.1f}x az parcacik')

print('\nUretilen dosyalar:')
!ls -la *.itp topol.top at2r_cg.pdb


> 🎓 **Elastic network neden gerekli?**
>
> Martini, proteinin üçüncül yapısını kendi başına koruyamaz. Çözüm: yapıyı bir
> yay ağıyla dışarıdan sabitlemek.
>
> **Bedeli:** proteininiz artık katlanamaz, büyük konformasyonel değişim yapamaz.
> Martini ile bir GPCR'ın aktivasyon geçişini **inceleyemezsiniz**.
> İnceleyebileceğiniz: lipid etkileşimleri, oligomerleşme, difüzyon.


In [ ]:
# Uretilen topolojiye bakalim - elastic network'un boyutu
import glob
itp = sorted(glob.glob('molecule_*.itp'))[0]
lines = open(itp).read().splitlines()
print('Dosya:', itp, '|', len(lines), 'satir\n')
for i, l in enumerate(lines):
    if l.strip().startswith('[') :
        print(f'{i:>6}  {l.strip()}')


---
## 4 · Martini 3 kuvvet alanı dosyalarını indir

`martinize2` proteinin topolojisini üretti, ama Martini'nin **genel** parametre
dosyaları (bead tipleri, lipidler, su, iyonlar) ayrıca gerekiyor.

Kaynak: [marrink-lab/martini-forcefields](https://github.com/marrink-lab/martini-forcefields)

> ℹ️ Ana dosya (`martini_v3.0.0.itp`) ~16 MB — indirmesi biraz sürebilir.


In [ ]:
BASE = 'https://raw.githubusercontent.com/marrink-lab/martini-forcefields/main/martini_forcefields/regular/v3.0.0/gmx_files'
files = [
    'martini_v3.0.0.itp',
    'martini_v3.0.0_solvents_v1.itp',
    'martini_v3.0.0_ions_v1.itp',
    'martini_v3.0.0_phospholipids_v1.itp',
]
for f in files:
    !wget -q {BASE}/{f} -O {f}
!ls -lh martini_v3.0.0*.itp


---
## 5 · `insane` — membran, su, iyon

| Bayrak | Ne yapar |
|---|---|
| `-box 12,12,14` | kutu boyutu (nm) |
| `-l POPC:1` | lipid kompozisyonu |
| `-sol W` | Martini standart suyu (1 bead ≈ 4 su molekülü) |
| `-salt 0.15` | 0.15 M NaCl |
| `-center` | proteini kutuya ortala |

> 💡 Karışık membran isterseniz: `-l POPC:7 -l POPE:2 -l CHOL:1`


In [ ]:
!insane \
  -f at2r_cg.pdb \
  -o sistem.gro \
  -p sistem_insane.top \
  -pbc square \
  -box 12,12,14 \
  -l POPC:1 \
  -sol W \
  -salt 0.15 \
  -center \
  -dm 0


In [ ]:
print('--- insane ne uretti? ---')
print(open('sistem_insane.top').read())

n = int(open('sistem.gro').read().splitlines()[1])
print(f'Toplam parcacik: {n:,}')


> 🔍 **Karşılaştırma anı.** Sabahki all-atom sistem kaç atomdu?
> Aynı hacimde all-atom bir sistem yaklaşık **10 kat** daha fazla parçacık içerirdi —
> üstelik Martini daha büyük zaman adımı da kullanabiliyor.


---
## 6 · Topolojiyi düzelt

**Burası en sık takınılan adımdır.**

`insane` bir `.top` üretir ama `#include` satırları eksik/yanlıştır — çünkü
`insane` proteininizin topolojisini bilmez. Doğrusunu biz yazıyoruz.

Sıralama önemli: önce genel kuvvet alanı, sonra moleküller.


In [ ]:
import re, glob

prot_itp = sorted(glob.glob('molecule_*.itp'))[0]

# insane'in urettigi [ molecules ] bolumunu al
raw = open('sistem_insane.top').read()
mols = raw.split('[ molecules ]')[1].strip().splitlines()
mols = [m for m in mols if m.strip() and not m.strip().startswith(';')]

# Protein satirinin adini martinize2'nin urettigi adla degistir
prot_name = None
for l in open(prot_itp):
    if l.strip().startswith('[ moleculetype ]'):
        continue
    if l.strip() and not l.strip().startswith((';','[')) and prot_name is None:
        prot_name = l.split()[0]
        break
print('Protein molekul adi:', prot_name)

fixed = []
for m in mols:
    parts = m.split()
    if parts[0].lower().startswith('protein'):
        fixed.append(f'{prot_name}   {parts[1]}')
    else:
        fixed.append(m.strip())

top = f'''; Biyofizik 2026 Kursu - Oturum 4
; AT2R (6JOD chain A) - Martini 3 - POPC membran

#include "martini_v3.0.0.itp"
#include "martini_v3.0.0_solvents_v1.itp"
#include "martini_v3.0.0_ions_v1.itp"
#include "martini_v3.0.0_phospholipids_v1.itp"
#include "{prot_itp}"

[ system ]
AT2R in POPC membrane (Martini 3)

[ molecules ]
''' + '\n'.join(fixed) + '\n'

open('sistem.top','w').write(top)
print('\n--- sistem.top ---')
print(top)


---
## 7 · `gmx grompp` ile doğrula

**Bu adım simülasyon koşmuyor.** Sadece şunu soruyor:
*koordinatlar, topoloji ve ayarlar birbiriyle tutarlı mı?*

`.tpr` dosyası üretilebiliyorsa input'unuz geçerlidir. Kursun asıl hedefi buydu.


In [ ]:
mdp = '''; Martini 3 - enerji minimizasyonu
integrator               = steep
nsteps                   = 500
emtol                    = 100
emstep                   = 0.01

nstlist                  = 20
cutoff-scheme            = Verlet
verlet-buffer-tolerance  = 0.005

coulombtype              = reaction-field
rcoulomb                 = 1.1
epsilon_r                = 15
vdw_type                 = cutoff
vdw-modifier             = Potential-shift-verlet
rvdw                     = 1.1
'''
open('minimization.mdp','w').write(mdp)

!gmx grompp -f minimization.mdp -c sistem.gro -p sistem.top -o em.tpr -maxwarn 5


### ✅ `em.tpr` oluştuysa başardınız.

Sabah GUI ile yaptığınız işin aynısını, bu kez komut satırından ve
kaba-taneli olarak yaptınız. **Ve bu sefer her adım tekrarlanabilir.**


In [ ]:
import os
if os.path.exists('em.tpr'):
    print('em.tpr olustu:', os.path.getsize('em.tpr'), 'bayt')
    print('\nInput hazir. Bu .tpr ile simulasyon koşulabilir.')
else:
    print('em.tpr yok - yukaridaki hata mesajini okuyalim.')


---
## 8 · *(opsiyonel)* Kısa enerji minimizasyonu

Vakit kalırsa birkaç saniye süren bir EM koşup sistemin çökmediğini görelim.


In [ ]:
!gmx mdrun -deffnm em -nsteps 200 -v 2>&1 | tail -20


---
## 🆘 Sık karşılaşılan hatalar

| Hata | Sebebi | Çözümü |
|---|---|---|
| `Atomtype X not found` | Kuvvet alanı `.itp`'si eksik | Bölüm 4'teki indirmeleri kontrol edin |
| `number of coordinates does not match topology` | `[ molecules ]` sayıları yanlış | Bölüm 6'yı tekrar çalıştırın |
| `Unknown molecule type Protein` | Protein adı `.top` ile `.itp`'de farklı | Bölüm 6 bunu otomatik düzeltir |
| `mkdssp not found` | DSSP kurulmamış | Bölüm 3'teki `-ss` alternatifini kullanın |
| `System has non-zero total charge` | Yuvarlama — normaldir | `-maxwarn` ile geçilebilir |
| `LINCS warning` / patlama | Sistem çakışmalı kurulmuş | `insane` kutu boyutunu büyütün |

---

## 📚 Bu notebook'un dayandığı kaynaklar

- [Martini Protein Model — Using Martinize2](https://cgmartini.nl/docs/tutorials/Martini3/ProteinsI/Tut1.html)
- [Modeling Complex Lipid Membranes — INSANE](https://cgmartini.nl/docs/tutorials/Martini3/LipidsII/)
- [Notes and Limitations](https://cgmartini.nl/docs/tutorials/Martini3/ProteinsI/Tut4.html)
- Kuvvet alanı dosyaları: [marrink-lab/martini-forcefields](https://github.com/marrink-lab/martini-forcefields)


---
## 💾 Çıktıları indirin

Colab oturumu kapanınca dosyalar silinir. Aşağıdaki hücre hepsini
tek bir `.zip` olarak bilgisayarınıza indirir.


In [ ]:
!zip -q -r oturum4_ciktilar.zip at2r.pdb at2r_cg.pdb molecule_*.itp sistem.gro sistem.top minimization.mdp em.tpr 2>/dev/null
from google.colab import files
files.download('oturum4_ciktilar.zip')
